📘 Lesson 2.2: Partial Derivatives, Loss Functions & Gradient Optimization.

Example Structure Data:
'''
Imagine we want to model how socio-economic satisfaction depends on GDP growth rate(x1) and Social trust(x2)
We will build a simple linear regression model (y^=W1X1 +W2X2)  using synthetic data based on European Social Survey (ESS) and Eurostat metrics:

import numpy as np

# Synthetic Dataset: 4 Countries [x1: GDP Growth %, x2: Social Trust (0-10)]
X = np.array([
    [1.5, 7.0],  # Country A gender male
    [3.0, 5.0],  # Country A gender female
    [0.5, 8.5],  # Country B gender male
    [2.0, 4.0]   # Country B gender female
], dtype=np.float64)

# Target: Overall Life Satisfaction (0-10 scale)
y = np.array([7.2, 6.1, 8.0, 5.0], dtype=np.float64)

Exercise:

Step 1: Prepare your data from your tables, for example get data about spain and germany and divide the samples by gender,
then chose data related with people trust and life satisfaction, the best variables that collects semanticly
this fact is Gini coeficient or Available family Brute Rent. 
Step 2: Implement MSE Loss: Create a function compute_mse(X, y, w)

Step 3: Implement Gradient Calculation: Create a function compute_gradient(X, y, w)
that computes and returns the analytical gradient vector ∇L(w)

Step 4: Write an optimization loop fit_gradient_descent(X, y, lr=0.01, epochs=100) that:
    -Initializes weights w^(0)=[0,0,0.0]
    -Iteratively updates weights using w^(t+1)= w^(t) -η∇L(w^(t))
    -Tracks and prints the loss every 20 epochs.
    -Returns the optimized weights  w* and the loss history.

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import random
#PATHING SCRIPT FOR EVERY EXCERSISE - PROJECT - WORKSPACE
# 1.Literal definition of route pathings(Every user of remote repository must config this pathing in order to find the local repository of his computer)
# My case:
ROOT = Path("/home/josu/Documentos/DataScienceCourse")

# We verify the existence before continue
if not ROOT.exists():
    raise FileNotFoundError(f"❌ The route {ROOT} doesn't exist. Check it.")

# 2. Fix the workspace
os.chdir(ROOT)
print(f"✅ Worskspace enabled!: {os.getcwd()}")

# 3. Define relative pathings to work properly
DATA_TABLES = ROOT / "data" / "raw" / "Tables"

# Let's verify our table folder:
if not DATA_TABLES.exists():
    # If it fails, monitorize the issue: 
    data_dir = ROOT / "data"
    if data_dir.exists():
        print(f"⚠️ the folder 'data' exists but it doesn't have any 'Tables'. 'data' content: {os.listdir(data_dir)}")
    else:
        print(f"⚠️ The folder 'data' doesn't exist on ROOT workspace: {os.listdir(ROOT)}")
    raise FileNotFoundError(f"❌ The tables route {DATA_TABLES} is missing.")

print(f"📂 Tables route detected: {DATA_TABLES}")

✅ Worskspace enabled!: /home/josu/Documentos/DataScienceCourse
📂 Tables route detected: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


In [2]:
#Step 1: Data Select and preparation.
#We decide to select GiniCoef as economic data for the exercice (https://es.wikipedia.org/wiki/Coeficiente_de_Gini)
#X1=GiniCoef(from Spain and Germany)
'''
# Gini Coefficient Summary:
# -------------------------
# Measures income/wealth inequality within a population.
# Range: 0 to 1 (unitless index).
# - 0: Perfect equality (everyone has the same income).
# - 1: Perfect inequality (one person has all the income).
# Note: Sometimes expressed as a percentage (0 to 100).
# Derived from the Lorenz curve; higher values indicate greater disparity.
'''


dfGini =pd.read_csv(DATA_TABLES/"GiniCoefEU.csv")

In [3]:
dfGini.info()

print(
    f"\n|DfGini content:\n{dfGini.head()}"
    f"\n\n|Var Description: \n{dfGini.describe()}"
)

<class 'pandas.DataFrame'>
RangeIndex: 870 entries, 0 to 869
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     870 non-null    str    
 1   LAST UPDATE  870 non-null    str    
 2   freq         870 non-null    str    
 3   age          870 non-null    str    
 4   statinfo     870 non-null    str    
 5   geo          870 non-null    str    
 6   TIME_PERIOD  870 non-null    int64  
 7   OBS_VALUE    870 non-null    float64
 8   OBS_FLAG     54 non-null     str    
 9   CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), str(7)
memory usage: 68.1 KB

|DfGini content:
              DATAFLOW        LAST UPDATE freq    age  statinfo geo  \
0  ESTAT:ILC_DI12(1.0)  08/06/26 23:00:00    A  TOTAL  GINI_HND  AL   
1  ESTAT:ILC_DI12(1.0)  08/06/26 23:00:00    A  TOTAL  GINI_HND  AL   
2  ESTAT:ILC_DI12(1.0)  08/06/26 23:00:00    A  TOTAL  GINI_HND  AL   
3  ESTAT:ILC_DI12(1.0)  08/06/26 23:00:00

In [4]:
#Lets filter the dataframe with the columns we want, grouping by
#discriminative values from 'geo'(countries), 'TIME_PERIOD', 'age'(it is calculated for ppl with less age than 18,
#and a global calculation for every age)
#At the same time we exclude other columns we dont want.
countries_gini: list[str]=['DE','ES']
year_gini = [2025]
age_gini: list[str]=['TOTAL']
#This vectorized one-line code applies a compound Boolean mask.
#Only rows satisfying all three conditions simultaneously are stored in dfGinif0.
#The column selection at the end of the expression also ensures that
#the new DataFrame contains only the three columns required for the analysis.
dfGinif0 = dfGini[dfGini["geo"].isin(countries_gini) & dfGini["TIME_PERIOD"].isin(year_gini) & dfGini["age"].isin(age_gini)][["geo", "TIME_PERIOD", "OBS_VALUE"]]

print(dfGinif0.head())

    geo  TIME_PERIOD  OBS_VALUE
89   DE         2025       30.1
150  ES         2025       30.8


In [5]:
#import gc
#Recomended to delete deprecated variables and dataframes from previous steps.
#We can use this one for delete all local variables '%reset -f'
#gc.collect()
del dfGini
del age_gini
del year_gini
del countries_gini

In [6]:
#In this case, Gini coeficient is expressed at OBS_VALUE column, 
#and its transformed in to float from 0.0 to 100.0
#Step1:
#Lets prepare the other 2 variables: X2=ppltrst(trust in others ppl from society),
# Y1=happy (overall life satisfaction)
# it will be filtered by gender and country reasons.
dfSurvey =pd.read_csv(DATA_TABLES/ "EssSurveys.csv")
dfSurvey.info()

print(
    f"\n|DfSurvey content:\n{dfSurvey.head()}"
    f"\n\n|Var Description: \n{dfSurvey.describe()}"
)


<class 'pandas.DataFrame'>
RangeIndex: 24602 entries, 0 to 24601
Data columns (total 31 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      24602 non-null  str    
 1   essround  24602 non-null  int64  
 2   edition   24602 non-null  float64
 3   proddate  24602 non-null  str    
 4   idno      24602 non-null  int64  
 5   cntry     24602 non-null  str    
 6   dweight   24602 non-null  float64
 7   pspwght   24602 non-null  float64
 8   pweight   24602 non-null  float64
 9   anweight  24602 non-null  float64
 10  netustm   24602 non-null  int64  
 11  ppltrst   24602 non-null  int64  
 12  trstplt   24602 non-null  int64  
 13  bctprd    24602 non-null  int64  
 14  prtdgcl   23008 non-null  float64
 15  lrscale   24602 non-null  int64  
 16  happy     24602 non-null  int64  
 17  aesfdrk   24602 non-null  int64  
 18  health    24602 non-null  int64  
 19  rlgdnm    24602 non-null  int64  
 20  rlgdgr    24602 non-null  int64  
 21  

In [7]:
#We will chose the values from 'ppltrst'(X2) and 'happy'(Y1)
#filtering for -gender column 'gndr'
#and country column 'cntry' via 'DE' and 'ES' values.
from pandas.core.common import random_state


random.seed(13)
country_surveys: list[str] = ['DE', 'ES']
gender_surveys:  list[int] = [1,2]
    # 1 = Male, 2 = Female, 9 = No answer (unclassified, bugged sample)
dfSurveyf0= dfSurvey[dfSurvey["cntry"].isin(country_surveys) 
                  & dfSurvey["gndr"].isin(gender_surveys)][["cntry", "gndr", "happy","ppltrst"]]

#After using a compound boolean mask, we apply a random sampling.
dfSurveyf1= dfSurveyf0.sample(n=150, random_state=13).reset_index(drop=True)

#Clear Cache and notebook variables.


#Lets get some information about DataSet

dfSurveyf1.info()
print(
    f"\n|DfSurveyf1 content:\n{dfSurveyf1.head()}"
    f"\n\n|Var Description: \n{dfSurveyf1.describe()}"
)              
del dfSurvey, dfSurveyf0, country_surveys, gender_surveys


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   cntry    150 non-null    str  
 1   gndr     150 non-null    int64
 2   happy    150 non-null    int64
 3   ppltrst  150 non-null    int64
dtypes: int64(3), str(1)
memory usage: 4.8 KB

|DfSurveyf1 content:
  cntry  gndr  happy  ppltrst
0    ES     2      8        7
1    DE     1      8        7
2    ES     1      8        8
3    DE     2      8        2
4    DE     1      9        6

|Var Description: 
             gndr       happy     ppltrst
count  150.000000  150.000000  150.000000
mean     1.506667    7.613333    5.020000
std      0.501630    1.690029    2.406563
min      1.000000    0.000000    0.000000
25%      1.000000    7.000000    3.000000
50%      2.000000    8.000000    5.000000
75%      2.000000    9.000000    7.000000
max      2.000000   10.000000   10.000000


In [8]:
#Lets clear the samples with no utility in this exercise,
#specificly the ordinal values attending to No Answer from
#respondent user.

'''
-happy 
77	Refusal*
88	Don't know*
99	No answer*

-ppltrst
77	Refusal*
88	Don't know*
99	No answer*
'''


happydelvalue: list[int]=[77,88,99]
#We use a vectorized NAND boolean mask in to the lines of the datasets.
#We can use isin refering the Not with ~ sign before the boolean mask statement
#For each column we filter

dfSurveyf2 = dfSurveyf1[
    (~dfSurveyf1["happy"].isin(happydelvalue))
    & (~dfSurveyf1["ppltrst"].isin(happydelvalue))
]


#Lets get some information about DataSet

dfSurveyf2.info()
print(
    f"\n|DfSurveyf2 content:\n{dfSurveyf2.head()}"
    f"\n\n|Var Description: \n{dfSurveyf2.describe()}"
)              



<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   cntry    150 non-null    str  
 1   gndr     150 non-null    int64
 2   happy    150 non-null    int64
 3   ppltrst  150 non-null    int64
dtypes: int64(3), str(1)
memory usage: 4.8 KB

|DfSurveyf2 content:
  cntry  gndr  happy  ppltrst
0    ES     2      8        7
1    DE     1      8        7
2    ES     1      8        8
3    DE     2      8        2
4    DE     1      9        6

|Var Description: 
             gndr       happy     ppltrst
count  150.000000  150.000000  150.000000
mean     1.506667    7.613333    5.020000
std      0.501630    1.690029    2.406563
min      1.000000    0.000000    0.000000
25%      1.000000    7.000000    3.000000
50%      2.000000    8.000000    5.000000
75%      2.000000    9.000000    7.000000
max      2.000000   10.000000   10.000000


In [9]:
#Lets reset de index setting the parameter to delete old index(drop=true)
#with a new ordered index, starting from zero
#We must select also (inplace=True) to allowing this command
#Overwrite in the existent dataframe
#dfSurveyf2.reset_index(drop=True, inplace=True)
del dfSurveyf1, happydelvalue
dfSurveyf2.head()

,cntry,gndr,happy,ppltrst
0,ES,2,8,7
1,DE,1,8,7
2,ES,1,8,8
3,DE,2,8,2
4,DE,1,9,6


In [ ]:
#STEP2:
#Lets create the arrays for implement MSE Lose, Gradient calculation, and Gradient Descent Loop:
#First location of the X array corresponds to Gini Coef applying a boolean mask
#discriminating by 'geo' that is the country
#Second location of X array will be the mean value of social trust filtering
#via boolean mask, discriminatig by 'cntry' and 'gndr' values.
#The four arrays of X , corresponds to 4 different samples, 1st:Country A('ES') is Spain and gender male(1) , 
#2nd: Country A and gender female(2)
#3d: Country B('DE') that is Germany and gender male
#4th: Conttry B and gender female.

X = np.array([
    [
        dfGinif0[dfGinif0["geo"] == "ES"]["OBS_VALUE"].iloc[0],
        dfSurveyf2[
            (dfSurveyf2["cntry"] == "ES")
            & (dfSurveyf2["gndr"] == 1)
        ]["ppltrst"].mean()
    ],

    [
        dfGinif0[dfGinif0["geo"] == "ES"]["OBS_VALUE"].iloc[0],
        dfSurveyf2[
            (dfSurveyf2["cntry"] == "ES")
            & (dfSurveyf2["gndr"] == 2)
        ]["ppltrst"].mean()
    ],

    [
        dfGinif0[dfGinif0["geo"] == "DE"]["OBS_VALUE"].iloc[0],
        dfSurveyf2[
            (dfSurveyf2["cntry"] == "DE")
            & (dfSurveyf2["gndr"] == 1)
        ]["ppltrst"].mean()
    ],

    [
        dfGinif0[dfGinif0["geo"] == "DE"]["OBS_VALUE"].iloc[0],
        dfSurveyf2[
            (dfSurveyf2["cntry"] == "DE")
            & (dfSurveyf2["gndr"] == 2)
        ]["ppltrst"].mean()
    ]
], dtype=np.float64)

print(X)

[[30.8         5.125     ]
 [30.8         4.94117647]
 [30.1         4.95238095]
 [30.1         5.07142857]]


In [ ]:
#2.1
#Lets define the corresponding 4 values of Y array 1D (one value for each subgroup already defined)
y = np.array([
    [
       dfSurveyf2[
            (dfSurveyf2["cntry"] == "ES")
            & (dfSurveyf2["gndr"] == 1)
       ]["happy"].mean()
    ],
    [
       dfSurveyf2[
            (dfSurveyf2["cntry"] == "ES")
            & (dfSurveyf2["gndr"] == 2)
       ]["happy"].mean()
    ],
    [
       dfSurveyf2[
            (dfSurveyf2["cntry"] == "DE")
            & (dfSurveyf2["gndr"] == 1)
       ]["happy"].mean()
    ],
    [
       dfSurveyf2[
            (dfSurveyf2["cntry"] == "DE")
            & (dfSurveyf2["gndr"] == 2)
       ]["happy"].mean()
    ],
    
    
    
])

print(
    f"\n|y array1 value:\n{y}"
    f"\n| Dimensions of X 2D array:\n{X.shape}"
    f"\n\n| Dimensions of y 1D array:\n{y.shape}"
)



|y array1 value:
[[7.875     ]
 [7.38235294]
 [7.52380952]
 [7.69047619]]
| Dimensions of X 2D array:
(4, 2)

| Dimensions of y 1D array:
(4, 1)


In [ ]:
#STEP 3:
#3.1 implement MSE Loss, create a function compute_mse(x,y,w) 
#that returns the scalar MSE loss for given weights (w)
